In [1]:
import pandas as pd
import numpy as np
import ast
import random
import numpy as np
from modules import DataFramePreprocessing, CreationDataset,CollateFunction,set_seed,seed_worker,EncodedHashedEmbeddings
import torch
from torch.nn import Embedding
import mmh3
from transformers import get_linear_schedule_with_warmup,AutoTokenizer,DistilBertModel
import itertools
import pyarrow.parquet as pq
from torch.utils.data import DataLoader



In [2]:
seed=42
set_seed(seed=seed)

## Creation of the processed dataframe

In [3]:
interactions_train_df=pd.read_csv("data/interactions_train.csv")
PP_recipes_df=pd.read_csv("data/PP_recipes.csv")
PP_users_df=pd.read_csv("data/PP_users.csv")
RAW_interactions_df=pd.read_csv("data/RAW_interactions.csv")
RAW_recipes_df=pd.read_csv("data/RAW_recipes.csv")


In [4]:
interactions_train_df_filtered=interactions_train_df.sample(n=50000,replace=False,random_state=42)

In [5]:
print(interactions_train_df_filtered.shape)


(50000, 6)


In [6]:
RAW_recipes_df.rename(columns={"id":"recipe_id"},inplace=True)
PP_recipes_df.rename(columns={"techniques":"techniques_recipes"},inplace=True)
PP_users_df.rename(columns={"techniques":"techniques_users"},inplace=True)

In [7]:
print("interactions_train_df_filtered columns:")
print(interactions_train_df_filtered.columns)
print("------------------------------------------------------------------")
print("PP_recipes_df columns:")
print(PP_recipes_df.columns)
print("-------------------------------------------------------------------")
print("PP_users_df columns:")
print(PP_users_df.columns)
print("----------------------------------------------------------------------")
print("RAW_interactions_df columns:")
print(RAW_interactions_df.columns)
print("------------------------------------------------------------------------")
print("RAW_recipes_df columns:")
print(RAW_recipes_df.columns)

interactions_train_df_filtered columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'u', 'i'], dtype='object')
------------------------------------------------------------------
PP_recipes_df columns:
Index(['id', 'i', 'name_tokens', 'ingredient_tokens', 'steps_tokens',
       'techniques_recipes', 'calorie_level', 'ingredient_ids'],
      dtype='object')
-------------------------------------------------------------------
PP_users_df columns:
Index(['u', 'techniques_users', 'items', 'n_items', 'ratings', 'n_ratings'], dtype='object')
----------------------------------------------------------------------
RAW_interactions_df columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'review'], dtype='object')
------------------------------------------------------------------------
RAW_recipes_df columns:
Index(['name', 'recipe_id', 'minutes', 'contributor_id', 'submitted', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'ingredients',
       'n_ingredients'],
      dty

In [8]:
full_df=interactions_train_df_filtered.merge(PP_recipes_df,on="i",how="left",suffixes=(None,"_1")).merge(PP_users_df,on="u",how="left",suffixes=(None,"_2")).merge(RAW_interactions_df,on=["user_id","recipe_id"],how="left",suffixes=(None,"_3")).merge(RAW_recipes_df,on=["recipe_id"],how="left",suffixes=(None,"_3"))

In [9]:
full_df=full_df.drop(columns=["review",'rating_3',"id","ingredients","submitted","contributor_id","date_3","date","steps_tokens","ingredient_tokens","name_tokens","u"],axis=1)

In [10]:
print(f"Columns of the full dataframe:")
print(full_df.columns)
print("----------------------------------------")
print(f"Shape of the full dataframe:")
print(full_df.shape)
print("----------------------------------------")
print(f"Columns of the filtered interaction dataframe:")
print(interactions_train_df_filtered.shape)
print("----------------------------------------")
print(f"Head of the full  dataframe:")
display(full_df.head())
print("----------------------------------------")
for col in full_df.columns:
    print(f"Unique values of {col} :")
    print(full_df[col].unique())
    print("----------------------------------------")
for col in full_df.columns:
    print(f"The type of {col} is")
    print(full_df[col].apply(type).unique())
    print("---------------------------------")

Columns of the full dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients'],
      dtype='object')
----------------------------------------
Shape of the full dataframe:
(50000, 20)
----------------------------------------
Columns of the filtered interaction dataframe:
(50000, 6)
----------------------------------------
Head of the full  dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,ratings,n_ratings,name,minutes,tags,nutrition,n_steps,steps,description,n_ingredients
0,231160,124810,5.0,170682,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[4574, 1910, 6906, 2499, 63, 332, 6270, 7470, ...","[3, 0, 0, 0, 1, 0, 0, 0, 0, 2, 1, 0, 0, 0, 0, ...","[8911, 20238, 117521, 79316, 154563, 34302, 17...",7,"[5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]",7,huckleberry or blueberry coffee cake,75,"['time-to-make', 'course', 'main-ingredient', ...","[174.1, 4.0, 92.0, 8.0, 7.0, 3.0, 11.0]",11,['beat margarine and cream cheese at medium sp...,"cooking light published this in their book, fi...",10
1,142629,31342,5.0,120865,"[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",1,"[1168, 1284, 8021, 2499, 330, 6270, 3217, 2318...","[6, 0, 0, 3, 1, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, ...","[122019, 150060, 117906, 82202, 119046, 141911...",10,"[4.0, 5.0, 5.0, 5.0, 4.0, 5.0, 3.0, 5.0, 4.0, ...",10,blender quiche or whatever you have in your ...,65,"['weeknight', 'time-to-make', 'course', 'main-...","[308.4, 37.0, 8.0, 20.0, 21.0, 41.0, 3.0]",10,"['preheat oven to 350f', 'generously grease a ...",the great part is you can use whatever leftove...,12
2,822358,441841,4.0,23142,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[1648, 1706, 3347, 5298, 3044, 4623, 2148, 6270]","[1, 1, 0, 2, 3, 0, 0, 0, 1, 5, 0, 1, 0, 0, 0, ...","[29761, 87944, 72370, 139097, 39470, 37755, 23...",17,"[4.0, 5.0, 5.0, 5.0, 4.0, 4.0, 4.0, 5.0, 5.0, ...",17,sweet orange coleslaw,10,"['15-minutes-or-less', 'time-to-make', 'course...","[290.1, 21.0, 71.0, 14.0, 7.0, 8.0, 13.0]",6,"['chop the apples', 'in a small bowl mix the o...",this recipe came from one of my mom's cookbook...,8
3,655579,211110,4.0,156541,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[7031, 5006, 339, 3203, 5975, 6270, 590]","[10, 0, 0, 5, 11, 0, 0, 0, 0, 7, 1, 3, 0, 0, 1...","[34492, 71279, 73805, 68721, 114795, 35164, 33...",28,"[2.0, 3.0, 4.0, 2.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...",28,wilted greens with garlic and balsamic vinegar,15,"['15-minutes-or-less', 'time-to-make', 'course...","[77.5, 10.0, 5.0, 5.0, 2.0, 4.0, 1.0]",5,"['cut the greens into strips 1 inch wide', 'in...",this simple but delicious side dish can be mad...,7
4,35140,114469,5.0,30839,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",2,"[1240, 6270, 5006, 1257, 2683, 5168, 800, 2461...","[176, 4, 2, 55, 72, 0, 1, 7, 4, 141, 13, 15, 0...","[121722, 36691, 151293, 154016, 1205, 12802, 1...",364,"[5.0, 5.0, 0.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...",364,chicken in the limelight,75,"['time-to-make', 'course', 'main-ingredient', ...","[645.9, 69.0, 11.0, 22.0, 80.0, 54.0, 4.0]",8,"['preheat oven to 350', 'grate lime peel and s...",very tender chicken soaked with flavor,9


----------------------------------------
Unique values of user_id :
[ 231160  142629  822358 ... 1086887 1364251  508459]
----------------------------------------
Unique values of recipe_id :
[124810  31342 441841 ...  13564 422436 398104]
----------------------------------------
Unique values of rating :
[5. 4. 3. 2. 0. 1.]
----------------------------------------
Unique values of i :
[170682 120865  23142 ... 110221  82789  70493]
----------------------------------------
Unique values of techniques_recipes :
['[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'
 '[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'
 '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [11]:
PP_users_df["len_items"]=PP_users_df["items"].str.count(r",")+1
PP_recipes_df["len_ingredients"]=PP_recipes_df["ingredient_ids"].str.count(r",")+1
max_len_ingredients=PP_recipes_df["len_ingredients"].max()
max_len_items=PP_users_df["len_items"].max()
PP_users_df.drop(columns=["len_items"],inplace=True)
PP_recipes_df.drop(columns=["len_ingredients"],inplace=True)

In [12]:
full_df_processed=DataFramePreprocessing(full_df,max_len_ingredients,max_len_items)
full_df_processed=full_df_processed.preprocessing()

In [4]:
PP_recipes_df["ingredient_ids_transformed"]=PP_recipes_df["ingredient_ids"].apply(lambda x: list(map(int,ast.literal_eval(x))))
all_ingredient_ids=np.unique(list(itertools.chain.from_iterable(list(PP_recipes_df["ingredient_ids_transformed"]))))
ingredient_continuous_ids=pd.Series([i for i in range(1,len(all_ingredient_ids)+1)],index=list(np.sort(all_ingredient_ids)))

In [14]:
def mapping_list(x:list,mapping_serie:pd.Series):
    return [mapping_serie.loc[i] if i in mapping_serie.index else 0 for i in x]

In [15]:
full_df_processed["ingredient_ids_continuous"]=full_df_processed["ingredient_ids"].apply(lambda x : mapping_list(x,ingredient_continuous_ids))

In [16]:
full_df_processed.to_parquet(path="./modules/dataframe/full_df_processed.parquet")

## Load of the dataframe

In [5]:
parquet_file = pq.ParquetFile("./modules/dataframe/full_df_processed.parquet")

chunks = []
for batch in parquet_file.iter_batches(batch_size=10000):
    chunks.append(batch.to_pandas())

full_df_processed = pd.concat(chunks, ignore_index=True)

In [6]:
print(f"Columns of the full processed dataframe:")
print(full_df_processed.columns)
print("----------------------------------------")
print(f"Shape of the full processed dataframe:")
print(full_df_processed.shape)
print("----------------------------------------")
print(f"Head of the full  processed dataframe:")
display(full_df_processed.head())
print("----------------------------------------")
for col in full_df_processed.columns:
    print(f"Exemple of {col} :")
    print(full_df_processed.iloc[0][col])
    print("----------------------------------------")
for col in full_df_processed.columns:
    print(f"The type of {col} is")
    print(full_df_processed[col].apply(type).unique())
    print("---------------------------------")

Columns of the full processed dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients',
       'ingredient_ids_continuous'],
      dtype='object')
----------------------------------------
Shape of the full processed dataframe:
(50000, 21)
----------------------------------------
Head of the full  processed dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,...,n_ratings,name,minutes,tags,nutrition,n_steps,steps,description,n_ingredients,ingredient_ids_continuous
0,231160,124810,5.0,170683,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[4574, 1910, 6906, 2499, 63, 332, 6270, 7470, ...","[3, 0, 0, 0, 1, 0, 0, 0, 0, 2, 1, 0, 0, 0, 0, ...","[8912, 20239, 117522, 79317, 154564, 34303, 17...",7,...,7,huckleberry or blueberry coffee cake,75,time to make [SEP] course [SEP] main ingredien...,"[174.1, 4.0, 92.0, 8.0, 7.0, 3.0, 11.0]",11,beat margarine and cream cheese at medium spee...,"cooking light published this in their book, fi...",10,"[4560, 1904, 6880, 2493, 63, 331, 6248, 7443, ..."
1,142629,31342,5.0,120866,"[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",1,"[1168, 1284, 8021, 2499, 330, 6270, 3217, 2318...","[6, 0, 0, 3, 1, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, ...","[122020, 150061, 117907, 82203, 119047, 141912...",10,...,10,blender quiche or whatever you have in your ...,65,weeknight [SEP] time to make [SEP] course [SEP...,"[308.4, 37.0, 8.0, 20.0, 21.0, 41.0, 3.0]",10,preheat oven to 350f [SEP] generously grease a...,the great part is you can use whatever leftove...,12,"[1163, 1278, 7992, 2493, 329, 6248, 3209, 2312..."
2,822358,441841,4.0,23143,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[1648, 1706, 3347, 5298, 3044, 4623, 2148, 627...","[1, 1, 0, 2, 3, 0, 0, 0, 1, 5, 0, 1, 0, 0, 0, ...","[29762, 87945, 72371, 139098, 39471, 37756, 23...",17,...,17,sweet orange coleslaw,10,15 minutes or less [SEP] time to make [SEP] co...,"[290.1, 21.0, 71.0, 14.0, 7.0, 8.0, 13.0]",6,chop the apples [SEP] in a small bowl mix the ...,this recipe came from one of my mom's cookbook...,8,"[1642, 1700, 3339, 5281, 3036, 4609, 2142, 624..."
3,655579,211110,4.0,156542,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[7031, 5006, 339, 3203, 5975, 6270, 590, 0, 0,...","[10, 0, 0, 5, 11, 0, 0, 0, 0, 7, 1, 3, 0, 0, 1...","[34493, 71280, 73806, 68722, 114796, 35165, 33...",28,...,28,wilted greens with garlic and balsamic vinegar,15,15 minutes or less [SEP] time to make [SEP] co...,"[77.5, 10.0, 5.0, 5.0, 2.0, 4.0, 1.0]",5,cut the greens into strips 1 inch wide [SEP] i...,this simple but delicious side dish can be mad...,7,"[7005, 4990, 338, 3195, 5953, 6248, 588, 1, 1,..."
4,35140,114469,5.0,30840,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",2,"[1240, 6270, 5006, 1257, 2683, 5168, 800, 2461...","[176, 4, 2, 55, 72, 0, 1, 7, 4, 141, 13, 15, 0...","[121723, 36692, 151294, 154017, 1206, 12803, 1...",364,...,364,chicken in the limelight,75,time to make [SEP] course [SEP] main ingredien...,"[645.9, 69.0, 11.0, 22.0, 80.0, 54.0, 4.0]",8,preheat oven to 350 [SEP] grate lime peel and ...,very tender chicken soaked with flavor,9,"[1235, 6248, 4990, 1252, 2677, 5151, 798, 2455..."


----------------------------------------
Exemple of user_id :
231160
----------------------------------------
Exemple of recipe_id :
124810
----------------------------------------
Exemple of rating :
5.0
----------------------------------------
Exemple of i :
170683
----------------------------------------
Exemple of techniques_recipes :
[1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
----------------------------------------
Exemple of calorie_level :
0
----------------------------------------
Exemple of ingredient_ids :
[4574 1910 6906 2499   63  332 6270 7470 3819 3497    0    0    0    0
    0    0    0    0    0    0]
----------------------------------------
Exemple of techniques_users :
[3 0 0 0 1 0 0 0 0 2 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 2 1 0 0 0 2 0 0 1
 0 0 0 0 0 1 2 0 0 0 1 0 0 0 0 0 0 1 0 0 1]
----------------------------------------
Exemple of items :
[  8912  20239 117522 ...      0      0      0]


## Model

In [7]:
dataset=CreationDataset(full_df_processed)

In [8]:
dataset.__getitem__(10)

{'user_id': 186070,
 'recipe_id': 11107,
 'rating': 4.0,
 'i': 3815,
 'techniques_recipes': array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int64),
 'calorie_level': 1,
 'ingredient_ids': array([7655, 5901, 2503,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0]),
 'techniques_users': array([86,  3,  1, 28, 24,  0,  0,  7,  0, 58,  7,  3,  0,  0,  2,  0, 23,
         0,  0,  7, 14,  3,  0, 11,  3,  0,  2,  5, 28,  9,  0,  0,  1, 37,
         0,  0, 20, 15,  8,  1,  0,  4,  8, 21,  1,  2, 17,  2,  0,  5,  0,
         0,  0,  6,  6, 18,  2, 20], dtype=int64),
 'items': array([ 80000, 111111,  61367, ...,      0,      0,      0]),
 'n_items': 174,
 'ratings': array([5, 5, 5, ..., 0, 0, 0]),
 'n_ratings': 174,
 'name': 'buttered egg noodles  best ever',
 'minutes': 10,
 'tags':

In [9]:
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [10]:
generator=torch.Generator()
generator.manual_seed(seed)
collate_function_object=CollateFunction(tokenizer=tokenizer)
collate_fn=collate_function_object.collate_fn
dataloader=DataLoader(dataset,batch_size=32,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=True,drop_last=True)

In [12]:
batch=next(iter(dataloader))
for key in list(batch.keys()):
    print(key)
    print(f"The shape of batch {key} is :")
    print(batch[key].shape)
    print("-------------------")

user_id
The shape of batch user_id is :
torch.Size([32, 1])
-------------------
recipe_id
The shape of batch recipe_id is :
torch.Size([32, 1])
-------------------
rating
The shape of batch rating is :
torch.Size([32, 1])
-------------------
i
The shape of batch i is :
torch.Size([32, 1])
-------------------
technique_recipes
The shape of batch technique_recipes is :
torch.Size([32, 58])
-------------------
calorie_level
The shape of batch calorie_level is :
torch.Size([32, 1])
-------------------
ingredient_ids
The shape of batch ingredient_ids is :
torch.Size([32, 20])
-------------------
techniques_users
The shape of batch techniques_users is :
torch.Size([32, 58])
-------------------
items
The shape of batch items is :
torch.Size([32, 6437])
-------------------
n_items
The shape of batch n_items is :
torch.Size([32, 1])
-------------------
ratings
The shape of batch ratings is :
torch.Size([32, 6437])
-------------------
n_ratings
The shape of batch n_ratings is :
torch.Size([32, 1

In [24]:
n_recipes_ids=len(np.unique(PP_recipes_df["i"]))
n_ingredients_ids=len(all_ingredient_ids)

print(f"There are {n_recipes_ids} recipe ids")
print(f"There are {n_ingredients_ids} ingredient ids")

There are 178265 recipe ids
There are 7993 ingredient ids


In [25]:
hash_encoder=EncodedHashedEmbeddings(n_recipes_ids,1024,n_ingredients_ids,1024)
hashed_recipes_ids_encoded_embeddings,hashed_ingredients_ids_encoded_embeddings=hash_encoder.get_encoded_hashed_embeddings("./modules/hashed_encoded_tables")

In [18]:
hashed_ingredients_ids_encoded_embeddings=torch.load("./modules/hashed_encoded_tables/hashed_embeddings_ingredients.pt")
hashed_recipes_ids_encoded_embeddings=torch.load("./modules/hashed_encoded_tables/hashed_embeddings_recipes.pt")

In [27]:
print(hashed_ingredients_ids_encoded_embeddings.shape)
print(hashed_recipes_ids_encoded_embeddings.shape)

torch.Size([7994, 1024])
torch.Size([178266, 1024])


In [33]:
print(hashed_recipes_ids_encoded_embeddings)

tensor([[ 0.7909, -0.9181,  0.4172,  ..., -0.1900,  0.4413,  0.9525],
        [-0.6644,  0.3263,  0.5654,  ...,  0.5458, -0.7577, -0.6355],
        [ 0.1152,  0.0698,  0.0535,  ..., -0.4404,  0.5246,  0.4121],
        ...,
        [ 0.9130, -0.5091, -0.2525,  ..., -0.9953, -0.0469, -0.3890],
        [ 0.0600, -0.6166, -0.7683,  ...,  0.6400, -0.9979,  0.9455],
        [ 0.0768, -0.9113,  0.0535,  ...,  0.2832, -0.0500,  0.4495]])


In [29]:
filter=[1,1,2,2]

In [30]:
print(hashed_recipes_ids_encoded_embeddings[batch["items"],:].shape)

torch.Size([32, 6437, 1024])


In [31]:
print(hashed_recipes_ids_encoded_embeddings[filter,:])

tensor([[-0.6644,  0.3263,  0.5654,  ...,  0.5458, -0.7577, -0.6355],
        [-0.6644,  0.3263,  0.5654,  ...,  0.5458, -0.7577, -0.6355],
        [ 0.1152,  0.0698,  0.0535,  ..., -0.4404,  0.5246,  0.4121],
        [ 0.1152,  0.0698,  0.0535,  ..., -0.4404,  0.5246,  0.4121]])


In [32]:
PP_users_df["items"]=PP_users_df["items"].apply(lambda x : list(map(int,ast.literal_eval(x))))
print(np.unique(list(itertools.chain.from_iterable(list(PP_users_df["items"])))))

[     0      1      2 ... 178257 178261 178262]


In [25]:
print(batch["items"].shape)
print(batch["items"])

torch.Size([32, 6437])
tensor([[125929, 135695, 147187,  ...,      0,      0,      0],
        [ 44203,   7481, 165528,  ...,      0,      0,      0],
        [134305, 169902, 109856,  ...,      0,      0,      0],
        ...,
        [ 49370,  84047, 114955,  ...,      0,      0,      0],
        [112382,  76064, 159609,  ...,      0,      0,      0],
        [ 28553,  34760, 151713,  ...,      0,      0,      0]],
       dtype=torch.int32)


In [29]:
print(hashed_recipes_ids_encoded_embeddings.shape)
print(hashed_recipes_ids_encoded_embeddings[batch["items"]].shape)

torch.Size([178266, 1024])
torch.Size([32, 6437, 1024])


In [30]:
print(batch["items"].shape)

torch.Size([32, 6437])


In [54]:
mask=(batch["items"]!=0).int()

In [55]:
print(mask)
print(mask.shape)

tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32)
torch.Size([32, 6437])


In [56]:
encoded=hashed_recipes_ids_encoded_embeddings[batch["items"]]
print(encoded.shape)

torch.Size([32, 6437, 1024])


In [57]:
mask=mask.unsqueeze(1)
print(mask.shape)
torch.mul(mask,encoded)

torch.Size([32, 1, 6437])


RuntimeError: The size of tensor a (6437) must match the size of tensor b (1024) at non-singleton dimension 2